# 04 · RSI2 MEAN REVERSION · RECONCILIATION
**Alphaca Capital** · `RSI2.qqq.v1` · variant `v1_qqq` · vehicle: Darwinex/NDX (QQQ proxy), D1 bars.

Reconciles the MT5 Expert Advisor (`RSI2_QQQ_Darwinex.mq5`) against `rsi2_engine.py`, layer by layer:

| Layer | EA file | What is compared with the engine |
| :--- | :--- | :--- |
| 1. Signals | `rsi2_log.csv` | One row per trading day: RSI2, SMA filter, BUY / SKIP / HOLD decision |
| 2. Trades | `rsi2_trades.csv` | One row per closed trade: entry / exit date, prices, net PnL, lots |

**How to produce the EA files**: run the Strategy Tester on a netting account, symbol NAS100 (or QQQ), D1,
every tick based on real ticks (or open prices for a first pass). The EA writes `rsi2_log.csv` and
`rsi2_trades.csv` to `Common/Files/`. Copy them to `data/Reconcilation/RSI2/` before running this notebook.

> **Note:** RSI2 trades daily bars. The engine signal fires at the **close of day i-1**;
> entry/exit happens at the **open of day i** (± `slip_bps_per_side`).  
> The EA logs the bar that *triggers* the entry (day i-1), while the fill occurs on day i —
> bear this in mind when matching timestamps.

In [ ]:
from pathlib import Path

VARIANT      = "v1_qqq"
HARD_LIMIT   = 0.08           # max portfolio drawdown (kill-switch reference)
CAPITAL      = 10_000.0       # fixed capital used in the Strategy Tester run

# EA capital mode: "fixed" = static equity; "compounding" = equity grows with P&L
EA_CAPITAL_MODE = "fixed"

RECON_DIR   = Path("../../data/Reconcilation/RSI2")
LOG_FILE    = RECON_DIR / "rsi2_log.csv"       # one row per trading day the EA processed
TRADES_FILE = RECON_DIR / "rsi2_trades.csv"    # one row per closed trade

## CONFIG

In [ ]:
import sys
import importlib
from pathlib import Path
import numpy as np
import pandas as pd

def find_upwards(name, start=None, must_be_dir=True):
    here = (start or Path.cwd()).resolve()
    for base in (here, *here.parents):
        target = base / name
        if target.is_dir() if must_be_dir else target.exists():
            return base
    raise FileNotFoundError(f"Could not find '{name}' walking up from {here}")

EDGE_DIR   = find_upwards("variants")
utils_path = str(find_upwards("utils") / "utils")
for p in (utils_path, str(EDGE_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import rsi2_engine as eng
import variant_config
from variant_config import load_variant

importlib.reload(eng)   # reset ENGINE_DEFAULTS so nothing leaks between variants

variant_config.EDGE_DIR = EDGE_DIR
assert (variant_config.EDGE_DIR / "variants").is_dir(), \
    f"variants/ not found under {variant_config.EDGE_DIR}"

cfg        = load_variant(VARIANT)
eng.set_engine_defaults(**cfg.get("engine_defaults", {}))

RUN_DIR    = cfg["run_dir"]
TICKER     = cfg["instrument"]["ticker"]          # "QQQ"
DATA_START = cfg["instrument"]["start"]           # "1999-01-01"
OOS_START  = cfg["oos_start_ts"]
SMA_LEN    = cfg["params"]["sma_len"]             # 230
RSI_THR    = cfg["params"]["rsi_thr"]             # 20.0
EXIT_MODE  = cfg["params"]["exit_mode"]           # "2green"
SLIP_BPS   = cfg["params"]["slip_bps_per_side"]   # 5.0

RULES = dict(rsi_thr=RSI_THR, sma_len=SMA_LEN, exit_mode=EXIT_MODE, slip_bps_per_side=SLIP_BPS)

print("Imports complete.")
print(f"Loaded rsi2_engine from: {EDGE_DIR / 'rsi2_engine.py'}")
print(f"Variant: {cfg['id']} | {TICKER} | RSI(2) < {RSI_THR:g} | SMA({SMA_LEN}) | "
      f"exit={EXIT_MODE} | {SLIP_BPS:g} bps/side")
print(f"Data source: {eng.ENGINE_DEFAULTS['data_source']}")

## 1. ENGINE REFERENCE
The engine runs on the full history with the frozen rules.
Signal dates are the **close dates that triggered** the entry (day i-1); fill dates are the
**open of the next trading day** (day i). The EA logs by signal date — we re-key accordingly.

In [ ]:
df_raw    = eng.load_data(TICKER, start=DATA_START)
df        = eng.prepare(df_raw, sma_len=SMA_LEN)
trades_all = eng.run_rsi2_backtest(df, **RULES)

# Reference table: one row per signal date (close of day i-1)
# signal_date = entry_time - 1 trading day  →  the date the EA logs the signal
entry_dates  = pd.to_datetime(trades_all["entry_time"])
signal_dates = entry_dates.map(
    lambda d: df.index[df.index.get_loc(d) - 1] if df.index.get_loc(d) > 0 else pd.NaT
)

ref = trades_all.copy()
ref["signal_date"]  = signal_dates.values
ref["entry_date"]   = entry_dates.values
ref["rsi_signal"]   = trades_all["rsi_signal"].values
ref["sma_at_entry"] = df["sma"].reindex(entry_dates).values
ref = ref.set_index("signal_date")

print(f"Engine trades (full history): {len(trades_all)}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
print(f"OOS start: {OOS_START.date()} | OOS trades: {(entry_dates >= OOS_START).sum()}")
print(trades_all[["entry_time","exit_time","rsi_signal","pnl","reason"]].tail(5).to_string())

## 2. SIGNAL RECONCILIATION (`rsi2_log.csv`)

The EA logs **every day** it evaluates the signal — both entries and skips.
Columns expected from the EA log (semicolon-separated, no header):

```
date ; rsi2 ; sma_close ; decision ; reason ; lots ; fill_price
```

- `decision`: `BUY` | `SKIP` | `HOLD` (position already open) | `EXIT`
- `reason`: why the signal was skipped (e.g. `rsi_above_thr`, `below_sma`, `in_position`)

We compare:
1. **RSI2 value** on signal days (tolerances: ≤ 0.5 RSI points)
2. **BUY/SKIP decision** on every day both sides evaluate (the EA must agree with the engine on every entry)

In [ ]:
if not LOG_FILE.exists():
    print(f"[SKIP] {LOG_FILE} not found.")
    print("Run the Strategy Tester with the EA and copy rsi2_log.csv to data/Reconcilation/RSI2/")
    log = None
else:
    log = pd.read_csv(LOG_FILE, sep=";", header=None,
                      names=["date", "rsi2_ea", "sma_ea", "decision", "reason", "lots", "fill_price"],
                      parse_dates=["date"])
    n_dup = int(log["date"].duplicated().sum())
    if n_dup:
        print(f"WARNING: {n_dup} duplicate dates in rsi2_log.csv — keeping last occurrence")
    log = log.drop_duplicates("date", keep="last").set_index("date").sort_index()
    log["rsi2_ea"]   = pd.to_numeric(log["rsi2_ea"],   errors="coerce")
    log["sma_ea"]    = pd.to_numeric(log["sma_ea"],    errors="coerce")
    log["fill_price"]= pd.to_numeric(log["fill_price"], errors="coerce")

    # Window: from first engine trade signal to last EA log entry
    w0 = ref.index[0]
    w1 = log.index.max()
    log_w = log[(log.index >= w0) & (log.index <= w1)]

    # Engine signal days within window
    ref_w = ref[(ref.index >= w0) & (ref.index <= w1)]
    buy_days = set(ref_w.index)

    # Merge on days the EA logged a BUY
    ea_buys = log_w[log_w["decision"] == "BUY"]
    eng_set  = buy_days
    ea_set   = set(ea_buys.index)

    only_eng = sorted(eng_set - ea_set)
    only_ea  = sorted(ea_set  - eng_set)

    print(f"Log window: {w0.date()} → {w1.date()} | {len(log_w)} rows")
    print(f"Engine BUY signals: {len(eng_set)} | EA BUY signals: {len(ea_set)}")
    print(f"Matched BUYs: {len(eng_set & ea_set)}")
    print(f"Only in engine: {len(only_eng)}", only_eng[:5] if only_eng else "")
    print(f"Only in EA:     {len(only_ea)}",  only_ea[:5]  if only_ea  else "")

    # RSI2 comparison on matched BUY days
    matched = sorted(eng_set & ea_set)
    if matched:
        m_log = log_w.loc[matched, "rsi2_ea"]
        m_eng = pd.Series(ref_w.loc[matched, "rsi_signal"].values, index=matched)
        d_rsi = (m_eng - m_log).abs()
        print(f"\nRSI2 diff on matched BUY days | max: {d_rsi.max():.4f} | mean: {d_rsi.mean():.4f}")
        if (d_rsi > 0.5).any():
            print("WARNING: RSI2 diff > 0.5 on dates:", sorted(d_rsi[d_rsi > 0.5].index))
        else:
            print("RSI2 values: OK (all within 0.5 RSI points)")

CHECKS = []
if log is not None:
    CHECKS.append(dict(section="R1 · Signals", name="BUY decision match",
                       status="PASS" if not only_eng and not only_ea else "FAIL",
                       detail=f"eng_only={len(only_eng)} ea_only={len(only_ea)}"))
    CHECKS.append(dict(section="R1 · Signals", name="RSI2 value tolerance <= 0.5",
                       status="PASS" if (d_rsi <= 0.5).all() else "FAIL",
                       detail=f"max diff={d_rsi.max():.4f}"))
else:
    CHECKS.append(dict(section="R1 · Signals", name="BUY decision match",
                       status="SKIP", detail="rsi2_log.csv not found"))
    CHECKS.append(dict(section="R1 · Signals", name="RSI2 value tolerance <= 0.5",
                       status="SKIP", detail="rsi2_log.csv not found"))

## 3. TRADE RECONCILIATION (`rsi2_trades.csv`)

Engine trades are matched with the EA closed trades by **entry date** (the fill day, day i).

Columns expected (semicolon-separated, no header):
```
time_in ; time_out ; price_in ; price_out ; lots ; profit ; swap ; commission
```

Checks:
* **Trade count**: same number of closed trades in the EA window
* **Entry price**: EA fill vs engine `open[i] × (1 + slip)` — tolerance 10 bps
* **Exit price**: EA fill vs engine `open[exit_day] × (1 − slip)` — tolerance 10 bps
* **Net PnL**: `(profit + swap + commission) / (price_in × lots)` vs engine `pnl / 100` — tolerance 20 bps
* **Lots**: only checked when `EA_CAPITAL_MODE == "fixed"` (compounding paths differ)

In [ ]:
TOL_PRICE_BPS = 10.0   # max acceptable fill gap in bps
TOL_PNL_BPS   = 20.0   # max acceptable net PnL gap in bps

if not TRADES_FILE.exists():
    print(f"[SKIP] {TRADES_FILE} not found.")
    deals = None
else:
    deals = pd.read_csv(TRADES_FILE, sep=";", parse_dates=["time_in", "time_out"])
    for col in ("swap", "commission"):
        if col not in deals.columns:
            deals[col] = 0.0

    # Normalise: use date only (RSI2 is daily; fill time is open of the day)
    deals["entry_date"] = deals["time_in"].dt.normalize()
    deals["exit_date"]  = deals["time_out"].dt.normalize()
    deals = deals.sort_values("entry_date").reset_index(drop=True)

    # Engine trades — keep only closed ones (not open_at_end)
    closed_eng = trades_all[trades_all["reason"] != "open_at_end"].copy()
    closed_eng["entry_date"] = pd.to_datetime(closed_eng["entry_time"]).dt.normalize()
    closed_eng["exit_date"]  = pd.to_datetime(closed_eng["exit_time"]).dt.normalize()

    # Restrict EA trades to the engine's date range
    ea_min = closed_eng["entry_date"].min()
    ea_max = closed_eng["entry_date"].max()
    deals_w = deals[(deals["entry_date"] >= ea_min) & (deals["entry_date"] <= ea_max)].copy()

    n_eng = len(closed_eng)
    n_ea  = len(deals_w)
    print(f"Engine closed trades: {n_eng} | EA trades in window: {n_ea}")

    CHECKS.append(dict(section="R2 · Trades", name="Trade count match",
                       status="PASS" if n_eng == n_ea else "FAIL",
                       detail=f"engine={n_eng} ea={n_ea}"))

    # Match by entry_date
    merged = closed_eng.merge(deals_w, on="entry_date", how="outer", indicator=True, suffixes=("_eng", "_ea"))
    only_eng_t = merged[merged["_merge"] == "left_only"]["entry_date"].tolist()
    only_ea_t  = merged[merged["_merge"] == "right_only"]["entry_date"].tolist()
    both       = merged[merged["_merge"] == "both"].copy()

    print(f"Matched: {len(both)} | Only engine: {len(only_eng_t)} | Only EA: {len(only_ea_t)}")
    if only_eng_t:
        print(f"  Trades only in engine: {[str(d.date()) for d in only_eng_t[:5]]}")
    if only_ea_t:
        print(f"  Trades only in EA: {[str(d.date()) for d in only_ea_t[:5]]}")

    bps = lambda a, b: ((a / b) - 1.0) * 1e4

    if len(both):
        both["d_entry_bps"] = bps(both["price_in"], both["entry_price_eng"]).abs()
        both["d_exit_bps"]  = bps(both["price_out"], both["exit_price_eng"]).abs()
        both["pnl_ea_pct"]  = (both["profit"] + both["swap"] + both["commission"]) / (both["price_in"] * both["lots"]) * 100.0
        both["d_pnl_bps"]   = ((both["pnl_ea_pct"] - both["pnl_eng"]) / both["entry_price_eng"].abs() * 1e4).abs()

        for metric, col, tol, name in [
            ("Entry price", "d_entry_bps", TOL_PRICE_BPS, "Entry price gap <= 10 bps"),
            ("Exit price",  "d_exit_bps",  TOL_PRICE_BPS, "Exit price gap <= 10 bps"),
            ("Net PnL",     "d_pnl_bps",   TOL_PNL_BPS,   "Net PnL gap <= 20 bps"),
        ]:
            worst = both[col].max()
            p95   = both[col].quantile(0.95)
            passed = worst <= tol
            print(f"{metric:15s} | worst={worst:.2f} bps | p95={p95:.2f} bps | {'OK' if passed else 'FAIL'}")
            CHECKS.append(dict(section="R2 · Trades", name=name,
                               status="PASS" if passed else "FAIL",
                               detail=f"worst={worst:.2f} bps p95={p95:.2f} bps"))

        # Lots check (only for fixed capital)
        if EA_CAPITAL_MODE == "fixed":
            lot_target = np.floor(CAPITAL * closed_eng["expo"] / closed_eng["entry_price"] / 0.01 + 1e-9) * 0.01
            lot_check  = (both["lots"].values - lot_target.values[:len(both)]).abs()
            lot_ok     = (lot_check <= 0.01).all()
            print(f"Lots diff        | max={lot_check.max():.4f} | {'OK' if lot_ok else 'FAIL'}")
            CHECKS.append(dict(section="R2 · Trades", name="Lots match (fixed capital)",
                               status="PASS" if lot_ok else "FAIL",
                               detail=f"max diff={lot_check.max():.4f} lots"))
        else:
            CHECKS.append(dict(section="R2 · Trades", name="Lots match (fixed capital)",
                               status="SKIP", detail="compounding mode — lots differ by design"))
else:
    for name in ("Trade count match", "Entry price gap <= 10 bps",
                 "Exit price gap <= 10 bps", "Net PnL gap <= 20 bps", "Lots match (fixed capital)"):
        CHECKS.append(dict(section="R2 · Trades", name=name,
                           status="SKIP", detail="rsi2_trades.csv not found"))

### 3a. WORST DEVIATIONS — DETAIL
Inspect the trades with the largest price or PnL gap to identify systematic vs sporadic issues.

In [ ]:
if deals is not None and len(both):
    worst5 = both.nlargest(5, "d_entry_bps")[
        ["entry_date", "entry_price_eng", "price_in", "d_entry_bps",
         "exit_price_eng", "price_out", "d_exit_bps", "pnl_eng", "pnl_ea_pct", "d_pnl_bps"]
    ].rename(columns={
        "entry_price_eng": "eng_entry", "price_in": "ea_entry",
        "exit_price_eng":  "eng_exit",  "price_out": "ea_exit",
        "pnl_eng": "pnl_eng%", "pnl_ea_pct": "pnl_ea%",
    })
    print("Top-5 worst entry deviations:")
    print(worst5.to_string(index=False))
else:
    print("[SKIP] No trade data loaded.")

## 4. KILL-SWITCH VERIFICATION
The EA has two kill switches:
- **Daily**: if the account drops ≥ `InpMaxDailyDD` % on a single day, the EA flattens and stops for the day.
- **Total**: if the account drops ≥ `InpMaxTotalDD` % from the starting equity, the EA stops permanently.

We verify that the EA **did not breach** the hard limit used in the validation (`HARD_LIMIT = 8%`).

In [ ]:
# Reconstruct equity curve from engine ATR sizing (matches the EA default InpRiskPercent = 0.33 %)
ret_atr = eng.daily_returns(df, trades_all, sizing="atr")

eq  = (1.0 + ret_atr).cumprod()
dd  = (eq / eq.cummax() - 1.0)
max_dd = float(dd.min())

print(f"Engine max drawdown (ATR sizing, full history): {max_dd:.2%}")
print(f"Hard limit: {HARD_LIMIT:.0%} | {'PASS — within limit' if max_dd >= -HARD_LIMIT else 'BREACH — exceeds limit'}")

CHECKS.append(dict(section="R3 · Risk", name=f"Engine max DD within hard limit ({HARD_LIMIT:.0%})",
                   status="PASS" if max_dd >= -HARD_LIMIT else "FAIL",
                   detail=f"max_dd={max_dd:.2%}"))

## 5. VERDICT
`RECONCILED` only when every check is `PASS` or `SKIP`.
Any `FAIL` must be explained and documented before the EA goes to the live account.

### Findings and documented exceptions
*(Fill in after running the tester and reviewing the checks.)*

- **Signals (R1):** — to be verified —
- **Trades (R2):** — to be verified —
- **Risk (R3):** — to be verified —

In [ ]:
status_df = pd.DataFrame(CHECKS)
print("=" * 70)
print("RECONCILIATION SUMMARY")
print("=" * 70)
for section, grp in status_df.groupby("section", sort=True):
    print(f"\n{section}")
    for _, r in grp.iterrows():
        print(f"   [{r['status']}] {r['name']}" + (f"  ->  {r['detail']}" if r["detail"] else ""))
n_fail = int((status_df["status"] == "FAIL").sum())
n_pass = int((status_df["status"] == "PASS").sum())
n_skip = int((status_df["status"] == "SKIP").sum())
print("\n" + "=" * 70)
print(f"PASS {n_pass} | FAIL {n_fail} | SKIP {n_skip}")
print("VERDICT: " + ("RECONCILED" if n_fail == 0 else
                      "REVIEW — the failed checks above must be explained before going live"))
print("=" * 70)

In [ ]:
import datetime as dt
print("Source file timestamps:")
for p in (LOG_FILE, TRADES_FILE):
    p = Path(p).resolve()
    if p.exists():
        st = p.stat()
        print(f"  {p.name}\n     modified: {dt.datetime.fromtimestamp(st.st_mtime):%Y-%m-%d %H:%M:%S}"
              f" | {st.st_size:,} bytes")
    else:
        print(f"  {p.name}  [NOT FOUND]")
print("\nNotebook run:", dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))